# 排队训练：v2、v2_1280，加上 H3 第 1、2 步的模型

一次 Run all，按顺序把下面的模型一个一个训完；每训完一个，ONNX 和记录马上存到 Drive 的 `mixed_dataset/versions/`。
**断线了就再 Run all 一次**：已经存好的会自动跳过，从没训完的那个重新开始。

配方全部是预锁定的 v0 配方（Ultralytics 8.4.182，YOLO11s，150 轮，seed 按表），只有影子模型 `v2_1280` 改 imgsz 1280 / batch 8（D6）。

**开始之前（在自己电脑上）**
1. `python dl.py pack-train --site penang`，把 `data/sites/penang/penang_train_all.zip` 上传到 Drive 的 `mixed_dataset/site/`（`pahang_train_all.zip` 已经在那里）。
2. Drive 的 `mixed_dataset/` 里要有 `merged_v5.zip` 和 `durian_11cls_640_v5_manifest.csv`（H3 的 S 模型要用 manifest 按组删图）。
3. 仓库已经 push（Colab 会 clone 它，要用里面的 `lock.json`）。

**运行时**：A100（Runtime → Change runtime type）。640 的每个大约一小时左右，1280 的一个半小时左右；整张表跑完要大半天，浏览器标签页别关。


In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/mixed_dataset"
REPO_URL = "https://github.com/xiaolin200206/Snapdurian_deploy.git"
ULTRALYTICS = "8.4.182"

# 顺序 = 优先级。v2 要在去 Klang 之前部署，所以排第一。
JOBS = [
    # 部署版本和它的 1280 影子（D6）
    dict(name="v2",      kind="version", sites=["pahang", "penang"], seed=0),
    dict(name="v2_1280", kind="version", sites=["pahang", "penang"], seed=0, imgsz=1280, batch=8),
    # H3 第 1 步（在 Penang 打分）：S1 = merged_v5 删掉和 Pahang 一样多的图、换成 Pahang；R0 = v0 配方的另外两个种子
    dict(name="S1_s0", kind="swap", base=[], new="pahang", seed=0),
    dict(name="S1_s1", kind="swap", base=[], new="pahang", seed=1),
    dict(name="S1_s2", kind="swap", base=[], new="pahang", seed=2),
    dict(name="R0_s1", kind="version", sites=[], seed=1),
    dict(name="R0_s2", kind="version", sites=[], seed=2),
    # H3 第 2 步（在 Klang 打分）：S2 = v1 的训练集删掉和 Penang 一样多的 merged_v5 图、换成 Penang；R1 = v1 配方的另外两个种子
    dict(name="S2_s0", kind="swap", base=["pahang"], new="penang", seed=0),
    dict(name="S2_s1", kind="swap", base=["pahang"], new="penang", seed=1),
    dict(name="S2_s2", kind="swap", base=["pahang"], new="penang", seed=2),
    dict(name="R1_s1", kind="version", sites=["pahang"], seed=1),
    dict(name="R1_s2", kind="version", sites=["pahang"], seed=2),
]
ONLY = None   # 只想跑其中几个就写名字，例如 ONLY = ["v2", "v2_1280"]；None = 整张表


In [ ]:
import os, sys, json, subprocess, shutil, zipfile, hashlib, time, gc, datetime as dt
from pathlib import Path
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("不在 Colab 里")
DRIVE = Path(DRIVE_DIR)
OUT = DRIVE / "versions"; OUT.mkdir(exist_ok=True)
REPO = Path("/content") / REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git")
if not REPO.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"ultralytics=={ULTRALYTICS}", "onnxruntime", "pyyaml", "onnx", "onnxslim"], check=True)
sys.path.insert(0, str(REPO)); os.chdir(REPO)
PRE = json.loads((REPO / "sequential" / "seq_prelock.json").read_text(encoding="utf-8"))
COMMIT = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 22), b""): h.update(b)
    return h.hexdigest()

def unzip_once(zp, dest, marker):
    if (dest / marker).exists():
        return
    print("解压", zp.name, "→", dest)
    local = Path("/content") / zp.name
    if not local.exists(): shutil.copy2(zp, local)
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local) as z: z.extractall(dest)

unzip_once(DRIVE / "merged_v5.zip", REPO / "data", "merged_v5")
MERGED = REPO / "data" / "merged_v5"
SITES_DIR = next((DRIVE / d for d in ("sites", "site") if (DRIVE / d).is_dir()), DRIVE / "sites")
jobs = [j for j in JOBS if ONLY is None or j["name"] in ONLY]

# 先把所有要用的东西检查一遍，别训到一半才发现缺文件
missing = []
for s in sorted({s for j in jobs for s in j.get("sites", []) + j.get("base", []) + ([j["new"]] if "new" in j else [])}):
    if not (SITES_DIR / f"{s}_train_all.zip").exists():
        missing.append(f"Drive 里没有 {SITES_DIR / (s + '_train_all.zip')}：电脑上 python dl.py pack-train --site {s}，再上传")
    if not (REPO / "sites" / s / "lock.json").exists():
        missing.append(f"GitHub 上的仓库里没有 sites/{s}/lock.json：电脑上先 git push")
if any(j["kind"] == "swap" for j in jobs) and not (DRIVE / "durian_11cls_640_v5_manifest.csv").exists():
    missing.append(f"Drive 里没有 {DRIVE / 'durian_11cls_640_v5_manifest.csv'}（H3 的 S 模型要用）")
assert not missing, "\n".join(missing)
for s in sorted({s for j in jobs for s in j.get("sites", []) + j.get("base", []) + ([j["new"]] if "new" in j else [])}):
    unzip_once(SITES_DIR / f"{s}_train_all.zip", REPO / "data" / "sites" / s / "train_all", "images")
if any(j["kind"] == "swap" for j in jobs):
    shutil.copy2(DRIVE / "durian_11cls_640_v5_manifest.csv", "/content/manifest.csv")

def done_file(j):
    sz = j.get("imgsz") or 640
    hits = sorted(OUT.glob(f"durian_11cls_{sz}_{j['name']}_*.onnx"))
    return hits[-1] if hits else None

for j in jobs:
    print(f"{j['name']:<9}", "✓ 已有 " + done_file(j).name if done_file(j) else "· 待训练")


In [ ]:
import torch
assert torch.cuda.is_available(), "没有 GPU：Runtime → Change runtime type → A100"
from ultralytics import YOLO
GPU = torch.cuda.get_device_name(0)

for j in jobs:
    name = j["name"]
    if done_file(j):
        print(f"跳过 {name}（Drive 里已经有 {done_file(j).name}）"); continue
    print(f"\n========== {name} ==========")
    if j["kind"] == "version":
        cmd = [sys.executable, "dl.py", "build-train", "--name", name, "--merged", str(MERGED), "--seed", str(j["seed"]), "--sites", *j["sites"]]
        if j.get("imgsz"): cmd += ["--imgsz", str(j["imgsz"])]
        if j.get("batch"): cmd += ["--batch", str(j["batch"])]
    else:
        cmd = [sys.executable, "dl.py", "build-swap", "--name", name, "--merged", str(MERGED), "--manifest", "/content/manifest.csv",
               "--new-site", j["new"], "--seed", str(j["seed"])] + (["--base-sites", *j["base"]] if j["base"] else [])
    subprocess.run(cmd, check=True)
    tdir = REPO / "data" / "train" / name
    kw = json.loads((tdir / "train_kwargs.json").read_text())
    t0 = time.time()
    model = YOLO(PRE["recipe_for_v1_v3"]["fixed_args"]["model"])
    model.train(**kw, project=str(DRIVE / "runs_seq"), device=0, workers=2)
    best = Path(model.trainer.best)
    hours = (time.time() - t0) / 3600
    best_sha = sha256(best)
    sz = kw.get("imgsz", 640)
    onnx_out = Path(YOLO(str(best)).export(format="onnx", opset=12, simplify=True, imgsz=sz, batch=1, dynamic=False))
    h = sha256(onnx_out)
    fname = f"durian_11cls_{sz}_{name}_{dt.date.today():%Y%m%d}_{h[:8]}.onnx"
    info = {"name": name, "job": j, "onnx_file": fname, "onnx_sha256": h, "best_pt_sha256": best_sha,
            "train_hours": round(hours, 2), "gpu": GPU, "repo_commit": COMMIT, "finished_utc": dt.datetime.utcnow().isoformat(timespec="seconds"),
            "build_info": json.loads((tdir / "build_info.json").read_text()), "train_kwargs": kw}
    (OUT / f"{name}_info.json").write_text(json.dumps(info, indent=2, ensure_ascii=False, default=str))
    if (tdir / "removed.txt").exists():
        shutil.copy2(tdir / "removed.txt", OUT / f"{name}_removed.txt")
    shutil.copy2(onnx_out, OUT / fname)          # ONNX 最后才拷：有它就代表这个模型完整做完了
    print(f"✓ {name}：{fname}  ({hours:.2f} h)")
    del model; gc.collect(); torch.cuda.empty_cache()


In [ ]:
# 全部做完以后跑这一格，把输出整段复制给 Claude（写进 versions_log / shadow_models / h3_models 用）
rows = []
for j in JOBS:
    f = OUT / f"{j['name']}_info.json"
    if f.exists():
        d = json.loads(f.read_text())
        rows.append(f"| {d['name']} | {d['onnx_file']} | {d['onnx_sha256']} | {d['best_pt_sha256']} | "
                    f"{d['finished_utc'][:10]}, {d['gpu'].replace('NVIDIA ', '')}, {d['train_hours']} h | "
                    f"{d['build_info'].get('n_train_images')} imgs, removed {d['build_info'].get('removed_from_merged_v5', 0)} |")
    else:
        rows.append(f"| {j['name']} | （还没训） | | | | |")
print("\n".join(rows))
